# 02 · Two AI agents, one catalog — governance in action

Both agents run the **same code**. The only difference is what peter granted in notebook
00. The governed tool `search_gallery` asks Lakekeeper to vend credentials for the Gold
Lance table; if the calling identity lacks `can_read_data`, Lakekeeper returns **no
credentials** and the agent is blind — the wall is in the catalog, not in app code.

In [1]:
import sys
sys.path.insert(0, '/work')
import ollama, lance
import mlib, gt, clip_util
from mlib import (NS_RAW, RAW_TABLE, NS_BRONZE, NS_SILVER, NS_GOLD, GOLD_TABLE,
                  OLLAMA_URL, CHAT_MODEL, ANALYST, CONTRACTOR, PIPELINE, get_token)
from icehelp import catalog

In [8]:
print(ANALYST, CONTRACTOR, PIPELINE)

('analyst-agent', 'analyst-secret-00000000000000000') ('contractor-agent', 'contractor-secret-0000000000000000') ('bootstrap', 'bootstrap-secret-0000000000000000')


## The governed tool

`search_gallery` embeds the query with CLIP, asks pylakekeeper to **vend** creds for the
Gold Lance table (`generic_tables.load(vended=True)`), opens it with Lance, and runs vector
search. If creds aren't vended it raises `AccessDenied` — the LLM only sees what this returns.

In [2]:
class AccessDenied(Exception): pass

def search_gallery(query, creds, k=4):
    qvec = clip_util.embed_text([query])[0]
    try:
        with gt.client(creds) as c:  # Lakekeeper vends creds only if this identity may read Gold
            t = c.generic_tables.load(NS_GOLD, GOLD_TABLE, vended=True)
        ds = lance.dataset(t.location, storage_options=t.lance_storage_options)
    except Exception as e:
        raise AccessDenied(f'{type(e).__name__}: {e}') from e
    return ds.to_table(nearest={'column': 'vector', 'q': qvec.tolist(), 'k': k},
                       columns=['object_id', 'title', 'theme', 'caption', 'image_uri']).to_pylist()

def compose_answer(query, hits):
    context = '\n'.join(f"- {h['title']}: {h['caption']}" for h in hits)
    prompt = (f'A user asked: {query!r}\n\nThese museum artworks were retrieved:\n{context}\n\n'
              'Answer in 2-3 sentences, referring to the specific artworks by title.')
    try:  # the chat model only writes prose; if it's missing/slow, fall back to the hits.
        ans = ollama.Client(host=OLLAMA_URL).generate(model=CHAT_MODEL, prompt=prompt).get('response', '').strip()
        if ans: return ans
    except Exception as e:
        print(f'  (chat model {CHAT_MODEL} unavailable: {type(e).__name__}; showing retrieved items)')
    return 'Retrieved: ' + '; '.join(f"{h['title']} ({h['theme']})" for h in hits)

def run_agent(query, creds):
    try:
        hits = search_gallery(query, creds)
    except AccessDenied as e:
        return {'allowed': False, 'error': str(e)}
    return {'allowed': True, 'hits': hits, 'answer': compose_answer(query, hits)}

## The analyst agent — granted `select` on Gold

In [3]:
QUERY = 'cats and other animals'
res = run_agent(QUERY, ANALYST)
if res['allowed']:
    print('ALLOWED — credentials vended.\n')
    for h in res['hits']: print(f"  • {h['title']}  [{h['theme']}]")
    print('\nagent answer:', res['answer'])
else:
    print('DENIED:', res['error'])

[2026-07-22T04:52:13Z WARN  lance::dataset::scanner] Deprecation warning, this behavior will change in the future. This search specified output columns but did not include `_distance`.  Currently the `_distance` column will be included.  In the future it will not.  Call `disable_scoring_autoprojection` to adopt the future behavior and avoid this warning


ALLOWED — credentials vended.

  • "The Concourse of the Birds", Folio 11r from a Mantiq al-Tayr (Language of the Birds)  [landscape]
  • A Woman with a Dog  [portrait]
  • A Peasant Family  [portrait]
  • Don Andrés de Andrade y la Cal  [portrait]

agent answer: The museum artworks featuring "cats and other animals" include "A Peasant Family" where children play with a cat, and "Don Andrés de Andrade y la Cal," which shows a dog at the side of a man.  However, "The Concourse of the Birds" provides a more fantastical scene filled with birds and a peacock, but not specifically cats.


## The contractor agent — same code, no grant on Gold

Lakekeeper returns **404 `NoSuchGenericTable`**, not 403: it won't even admit the table
exists to a principal that can't see it. Either way, no credentials are vended.

In [12]:
res = run_agent(QUERY, CONTRACTOR)
if res['allowed']:
    print('ALLOWED (unexpected!)')
else:
    print('DENIED by Lakekeeper — no credentials vended:\n ', res['error'])

DENIED by Lakekeeper — no credentials vended:
  NotFoundError: HTTP 404 (GET http://lakekeeper:8181/lakekeeper/v1/5d3986b4-8588-11f1-a99a-9fa10b7cf49d/namespaces/gold/generic-tables/image_embeddings): {"error":{"message":"Error getting tabular from catalog","type":"NoSuchGenericTableException","code":404,"stack":["Error ID: 019f8833-d064-7160-9b51-c85ff25cda29"]}}


## The boundary runs *through* the medallion

The contractor isn't locked out of everything — it's a legitimate collaborator. It **can**
read the source layers (raw image objects, metadata, captions); Lakekeeper withholds only
the Gold embeddings. Proof — the contractor fetches a raw image object and reads the Silver
captions just fine:

In [5]:
cat = catalog(get_token(*CONTRACTOR))
bronze = cat.load_table(f'{NS_BRONZE}.artworks').scan().to_arrow().to_pylist()
silver = cat.load_table(f'{NS_SILVER}.artwork_features').scan().to_arrow().to_pylist()
with gt.client(CONTRACTOR) as c:  # contractor CAN vend raw
    fs = gt.s3fs(c.generic_tables.load(NS_RAW, RAW_TABLE, vended=True))
img = gt.read_object(fs, bronze[0]['image_uri'])
print(f'contractor fetched a {len(img)}-byte raw image object and read {len(silver)} Silver captions — allowed.\n')
for r in silver[:5]: print(f"  • {r['title'][:40]!r}: {r['caption'][:60]!r}")
print('\n...but Gold (the embeddings) stays invisible to it. The wall is at Gold, per-layer.')

contractor fetched a 111536-byte raw image object and read 40 Silver captions — allowed.

  • 'The Penitence of Saint Jerome': 'Three-panel oil on canvas portrait of a man kneeling in fron'
  • 'Stela of the Steward Mentuwoser': 'urn with writing on it, people in front of it'
  • 'Sabine Houdon (1787–1836)': 'urn of a child in a white bust form against a gray backgroun'
  • 'Marie Antoinette in a Park': 'urns on a table in front of a tree with leaves.'
  • 'Young Woman with a Pink': 'urn of flowers on table with woman in red dress holding one'

...but Gold (the embeddings) stays invisible to it. The wall is at Gold, per-layer.


## Try it yourself — the governed chat widget

Type any query and ask **both** agents at once. The analyst answers from Gold; the
contractor is denied. Thumbnails come from `raw.images` (operator view, for display).

In [6]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Thumbnails: read raw image objects via the pipeline account (operator view).
with gt.client(PIPELINE) as c:
    _fs = gt.s3fs(c.generic_tables.load(NS_RAW, RAW_TABLE, vended=True))
def _thumb(uri):
    try: return gt.read_object(_fs, uri) if uri else None
    except Exception: return None

box = widgets.Text(value='cats and other animals', description='Ask:',
                   layout=widgets.Layout(width='70%'))
btn = widgets.Button(description='Ask both agents', button_style='primary')
out = widgets.Output()

def _render(label, creds):
    print(f'===== {label} =====')
    res = run_agent(box.value, creds)
    if not res['allowed']:
        print('  DENIED by Lakekeeper:', res['error']); print(); return
    print('  ' + res['answer'] + '\n')
    thumbs = [widgets.Image(value=b, format='jpg', width=110)
              for b in (_thumb(h.get('image_uri')) for h in res['hits']) if b]
    if thumbs: display(widgets.HBox(thumbs))
    for h in res['hits']: print(f"   • {h['title']} [{h['theme']}]")
    print()

def on_click(_):
    with out:
        clear_output()
        _render('analyst-agent (granted Gold)', ANALYST)
        _render('contractor-agent (denied Gold)', CONTRACTOR)

btn.on_click(on_click)
display(widgets.VBox([widgets.HBox([box, btn]), out]))

## (Optional) flip the grant live

The governance lever is one grant. Log in as **peter** and grant the contractor read on
Gold — then re-run the contractor cell above and watch it succeed. Revoke to restore.
```python
peter = mlib.device_login()
gold_id = mlib.namespace_id(peter.token, NS_GOLD)
cid = mlib.agent_user_id(CONTRACTOR)
mlib.grant_namespace(peter.token, gold_id, cid, 'select')                 # grant
# mlib.grant_namespace(peter.token, gold_id, cid, 'select', revoke=True)  # revoke
```

In [10]:
# Uncomment to pull the lever live (opens a browser login as peter):
peter = mlib.device_login()
gold_id = mlib.namespace_id(peter.token, NS_GOLD)
mlib.grant_namespace(peter.token, gold_id, mlib.agent_user_id(CONTRACTOR), 'select')
print(run_agent(QUERY, CONTRACTOR)['allowed'])  # -> True after the grant

Open this URL in your browser and approve the login:

    http://localhost:30080/realms/iceberg/device?user_code=RRWR-UFLF

(verification code: RRWR-UFLF)



✓ logged in as peter


[2026-07-22T05:01:25Z WARN  lance::dataset::scanner] Deprecation warning, this behavior will change in the future. This search specified output columns but did not include `_distance`.  Currently the `_distance` column will be included.  In the future it will not.  Call `disable_scoring_autoprojection` to adopt the future behavior and avoid this warning


True


---
**The takeaway:** the access wall lives in Lakekeeper's credential-vending layer, not in
application code. A denied agent gets no keys and *cannot* read the data — even running the
exact same code as the allowed one, and even for a Lance vector table.